In [ ]:
"""
NOTEBOOK: MODEL PERFORMANCE TRACKING
=====================================
Purpose: Monitor model performance over time
Output: Performance drift detection
"""

# 📈 Model Performance Tracking Notebook

**Objective:** Detect when models need retraining

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os

class PerformanceTracker:
    """Track model performance metrics over time"""
    
    def __init__(self):
        self.metrics_history = []
    
    def log_metrics(self, model_name, accuracy, timestamp=None):
        """Log model performance metrics"""
        if timestamp is None:
            timestamp = datetime.now()
        
        self.metrics_history.append({
            'model': model_name,
            'accuracy': accuracy,
            'timestamp': timestamp
        })
    
    def detect_drift(self, window_days=30, threshold=0.05):
        """Detect performance drift"""
        df = pd.DataFrame(self.metrics_history)
        
        if len(df) < 10:
            return {'drift_detected': False, 'message': 'Insufficient data'}
        
        # Calculate rolling average
        df['rolling_avg'] = df['accuracy'].rolling(window=7).mean()
        
        # Check for significant drop
        recent_avg = df.tail(window_days)['accuracy'].mean()
        historical_avg = df.head(-window_days)['accuracy'].mean() if len(df) > window_days else recent_avg
        
        drift = historical_avg - recent_avg
        
        if drift > threshold:
            return {
                'drift_detected': True,
                'drift_magnitude': drift,
                'message': f'Performance dropped by {drift:.2%} - Retraining recommended'
            }
        else:
            return {
                'drift_detected': False,
                'drift_magnitude': drift,
                'message': 'Model performance stable'
            }
    
    def plot_performance_trend(self):
        """Visualize performance over time"""
        df = pd.DataFrame(self.metrics_history)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for model in df['model'].unique():
            model_data = df[df['model'] == model]
            ax.plot(model_data['timestamp'], model_data['accuracy'], 
                   marker='o', label=model, linewidth=2)
        
        ax.axhline(y=0.75, color='red', linestyle='--', label='Target Threshold')
        ax.set_title('Model Performance Over Time', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel('Accuracy')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        os.makedirs('../../proofs', exist_ok=True)
        plt.savefig('../../proofs/performance_tracking.png', dpi=150)
        plt.show()

# Simulate performance tracking
tracker = PerformanceTracker()

# Log historical performance
dates = pd.date_range(start='2024-01-01', end=datetime.now(), freq='W')
for i, date in enumerate(dates):
    # Simulate gradual performance degradation
    accuracy = 0.85 - (i * 0.002)
    tracker.log_metrics('risk_model', accuracy, date)
    tracker.log_metrics('price_model', 0.88 - (i * 0.001), date)
    tracker.log_metrics('fraud_model', 0.92 - (i * 0.0005), date)

# Check for drift
drift_report = tracker.detect_drift()
print(f"📊 Drift Detection: {drift_report['message']}")

print("\n✅ Performance tracking complete!")